# 8.15 — Decoding strategies

Decoding is where a language model's probability distribution becomes a single sentence, so the sampling rule becomes part of the model's behavior.

A Transformer gives next-token logits and decoding chooses the sentence. Greedy, beam, temperature, top-k, and nucleus rules trade determinism, diversity, and constraint satisfaction.

Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 713
random.seed(SEED)
np.random.seed(SEED)


def softmax(values):
    arr = np.array(values, dtype=float)
    shifted = arr - np.max(arr)
    exp = np.exp(shifted)
    return exp / exp.sum()


def accuracy(pred, gold):
    return sum(int(a == b) for a, b in zip(pred, gold)) / max(1, len(gold))


def show_table(rows, headers):
    widths = [len(str(h)) for h in headers]
    for row in rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))
    print(" | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers)))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(str(value).ljust(widths[i]) for i, value in enumerate(row)))


def make_sequence_ladder(topic):
    if topic == "8.13":
        return [
            {"name": "D1 reversal", "sequences": [["dog", "bites"], ["bites", "dog"]], "labels": [1, 0], "max_pos": 2},
            {"name": "D2 clean order", "sequences": [["a", "then", "b"], ["b", "then", "a"], ["red", "before", "blue"], ["blue", "before", "red"], ["left", "right"]], "labels": [1, 0, 1, 0, 1], "max_pos": 5},
            {"name": "D3 agreement", "sequences": [["dogs", "chase", "cat"], ["cat", "chase", "dogs"], ["he", "likes", "tea"], ["tea", "likes", "he"], ["birds", "fly", "today"]], "labels": [1, 0, 1, 0, 1], "max_pos": 8},
            {"name": "D4 pair tasks", "sequences": [["subject", "verb", "object", "now"], ["object", "verb", "subject", "now"], ["query", "key", "value", "end"], ["value", "key", "query", "end"], ["first", "middle", "last", "stop"]], "labels": [1, 0, 1, 0, 1], "max_pos": 10},
            {"name": "D5 beyond learned", "sequences": [["start"] + ["filler"] * n + ["end"] for n in [4, 8, 12, 16, 20]], "labels": [1, 1, 1, 1, 1], "max_pos": 22},
        ]
    if topic == "8.14":
        return [
            {"name": "D1 mask toy", "T": 32, "w": 2, "evidence": [(19, 0)], "global_tokens": []},
            {"name": "D2 local deps", "T": 24, "w": 2, "evidence": [(4, 3), (8, 6), (12, 11), (16, 15), (20, 18)], "global_tokens": []},
            {"name": "D3 global random", "T": 40, "w": 2, "evidence": [(30, 0), (35, 1), (21, 18), (10, 8), (39, 3)], "global_tokens": [0, 1]},
            {"name": "D4 document snippets", "T": 64, "w": 3, "evidence": [(50, 2), (60, 4), (32, 31), (12, 10), (45, 5)], "global_tokens": [0, 2, 4]},
            {"name": "D5 far evidence", "T": 96, "w": 3, "evidence": [(90, 0), (88, 6), (70, 9), (44, 43), (30, 27)], "global_tokens": [0, 6, 9]},
        ]
    if topic == "8.15":
        return [
            {"name": "D1 logits", "rows": [[3.0, 2.0, 0.5]], "target": ["A"]},
            {"name": "D2 clean rows", "rows": [[3, 1, 0], [0, 3, 1], [1, 0, 3], [4, 2, 0], [0, 4, 2]], "target": list("ABCAB")},
            {"name": "D3 traps", "rows": [[2.1, 2.0, 0], [1.8, 1.7, 1.6], [3, 2.9, 0], [2, 2, 1.9], [0, 2.2, 2.1]], "target": list("BACCB")},
            {"name": "D4 candidates", "rows": [[2.5, 2.4, 0.1], [0.5, 2.0, 1.9], [2.2, 2.1, 2.0], [3.0, 0.2, 0.1], [0.1, 2.8, 2.7]], "target": list("ABABB")},
            {"name": "D5 long generation", "rows": [[2.0, 1.9, 0.2], [1.5, 1.4, 2.2], [2.1, 2.0, 0.4], [0.4, 2.2, 2.1], [1.9, 1.8, 0.7], [0.5, 2.4, 2.3], [2.3, 2.2, 0.2]], "target": list("ACABABB")},
        ]
    if topic == "8.16":
        return [
            {"name": "D1 lesson pair", "cand": "the cat sat", "ref": "the cat slept", "probs": [0.5, 0.25, 0.25]},
            {"name": "D2 paraphrases", "pairs": [("small cat sleeps", "small cat sleeps"), ("red bird flies", "red bird sings"), ("quick dog runs", "dog runs quickly"), ("bright moon rises", "moon rises bright"), ("blue car stops", "blue bus stops")]},
            {"name": "D3 predicate order", "pairs": [("the cat chased mouse", "the mouse chased cat"), ("doctor treats patient", "doctor helps patient"), ("team wins final", "team loses final"), ("user resets password", "password resets user"), ("model answers question", "model answers query")]},
            {"name": "D4 summaries", "pairs": [("sales rose in june", "sales rose in june after ads"), ("server failed at noon", "server recovered at noon"), ("policy changed for users", "policy changed for all users"), ("campaign reached creators", "campaign reached many creators"), ("video views increased", "video watch time increased")]},
            {"name": "D5 long refs", "pairs": [("the model answered safely with citations", "the model answered correctly with citations"), ("the summary covered three facts and omitted risk", "the summary covered three facts and named risk"), ("translation kept names but changed tense", "translation kept names and preserved tense"), ("caption mentions dog running near beach", "caption mentions dog walking near beach"), ("assistant refused unsafe request politely", "assistant refused unsafe request clearly")]},
        ]
    if topic == "8.17":
        return [
            {"name": "D1 Ada", "tokens": ["Ada", "works", "at", "OpenAI"], "tags": ["B-PER", "O", "O", "B-ORG"]},
            {"name": "D2 clean entities", "examples": [(["Ada", "joined", "OpenAI"], ["B-PER", "O", "B-ORG"]), (["Lin", "visits", "Paris"], ["B-PER", "O", "B-LOC"]), (["DeepMind", "hired", "Ilya"], ["B-ORG", "O", "B-PER"]), (["Mira", "met", "Sam"], ["B-PER", "O", "B-PER"]), (["Google", "opened", "office"], ["B-ORG", "O", "O"])]},
            {"name": "D3 adjacent padding", "examples": [(["Ada", "Bob", "spoke"], ["B-PER", "B-PER", "O"]), (["New", "York", "Labs"], ["B-LOC", "I-LOC", "B-ORG"]), (["Eve", "from", "IBM", "arrived"], ["B-PER", "O", "B-ORG", "O"]), (["Paris", "Paris", "team"], ["B-LOC", "B-ORG", "O"]), (["Zed", "Corp", "LLC"], ["B-ORG", "I-ORG", "I-ORG"])]},
            {"name": "D4 snippets", "examples": [(["Dr", "Chen", "at", "Mayo"], ["B-PER", "I-PER", "O", "B-ORG"]), (["Acme", "bought", "Beta"], ["B-ORG", "O", "B-ORG"]), (["Nina", "left", "Berlin"], ["B-PER", "O", "B-LOC"]), (["OpenAI", "San", "Francisco"], ["B-ORG", "B-LOC", "I-LOC"]), (["Raj", "from", "UN"], ["B-PER", "O", "B-ORG"])]},
            {"name": "D5 long multi", "examples": [(["Ada", "Lovelace", "worked", "with", "Babbage", "Labs"], ["B-PER", "I-PER", "O", "O", "B-ORG", "I-ORG"]), (["Maya", "from", "OpenAI", "visited", "London"], ["B-PER", "O", "B-ORG", "O", "B-LOC"]), (["IBM", "and", "Google", "met", "in", "Paris"], ["B-ORG", "O", "B-ORG", "O", "O", "B-LOC"]), (["Dr", "Lee", "joined", "Mayo", "Clinic"], ["B-PER", "I-PER", "O", "B-ORG", "I-ORG"]), (["Sam", "Altman", "visited", "New", "York"], ["B-PER", "I-PER", "O", "B-LOC", "I-LOC"])]},
        ]
    return [
        {"name": "D1 morphology", "tokens": ["walk", "walked", "walking"], "tags": ["VERB", "VERB", "VERB"]},
        {"name": "D2 clean POS", "examples": [(["dogs", "walk"], ["NOUN", "VERB"]), (["they", "walked"], ["NOUN", "VERB"]), (["walking", "helps"], ["NOUN", "VERB"]), (["red", "dogs", "run"], ["ADJ", "NOUN", "VERB"]), (["cats", "sleep"], ["NOUN", "VERB"])]},
        {"name": "D3 ambiguous", "examples": [(["walk", "home"], ["VERB", "NOUN"]), (["the", "walk"], ["DET", "NOUN"]), (["walking", "tour"], ["ADJ", "NOUN"]), (["we", "duck"], ["NOUN", "VERB"]), (["duck", "soup"], ["NOUN", "NOUN"])]},
        {"name": "D4 tagged lines", "examples": [(["bright", "birds", "sing"], ["ADJ", "NOUN", "VERB"]), (["users", "liked", "ads"], ["NOUN", "VERB", "NOUN"]), (["fast", "models", "serve"], ["ADJ", "NOUN", "VERB"]), (["the", "running", "joke"], ["DET", "ADJ", "NOUN"]), (["creators", "earned", "money"], ["NOUN", "VERB", "NOUN"])]},
        {"name": "D5 agreement", "examples": [(["the", "dogs", "were", "walking"], ["DET", "NOUN", "VERB", "VERB"]), (["a", "walk", "was", "short"], ["DET", "NOUN", "VERB", "ADJ"]), (["walking", "brands", "sell"], ["ADJ", "NOUN", "VERB"]), (["names", "ending", "ing", "confuse"], ["NOUN", "VERB", "NOUN", "VERB"]), (["past", "users", "walked"], ["ADJ", "NOUN", "VERB"])]},
    ]



TOKENS = ["A", "B", "C"]


def temperature_probs(logits, temperature=1.0):
    return softmax(np.array(logits, dtype=float) / temperature)


def top_k_probs(probs, k):
    probs = np.array(probs, dtype=float)
    keep = np.argsort(probs)[-k:]
    out = np.zeros_like(probs)
    out[keep] = probs[keep]
    out = out / out.sum()
    return out


def nucleus_probs(probs, threshold):
    probs = np.array(probs, dtype=float)
    order = list(np.argsort(-probs))
    keep = []
    total = 0.0
    for idx in order:
        keep.append(idx)
        total += float(probs[idx])
        if total >= threshold:
            break
    out = np.zeros_like(probs)
    out[keep] = probs[keep]
    out = out / out.sum()
    return out, keep


def beam_log_score(token_probs):
    return sum(math.log(p) for p in token_probs)


def decode(logits, strategy="greedy", rng=None):
    rng = rng or random.Random(SEED)
    probs = temperature_probs(logits)
    if strategy == "greedy":
        return TOKENS[int(np.argmax(probs))]
    if strategy == "top_k":
        probs = top_k_probs(probs, 2)
    if strategy == "nucleus":
        probs, keep = nucleus_probs(probs, 0.9)
    draw = rng.random()
    total = 0.0
    for token, prob in zip(TOKENS, probs):
        total += float(prob)
        if draw <= total:
            return token
    return TOKENS[-1]


def constrained_success(row, gold, index):
    if index % 2 == 0:
        pred = decode(row, "greedy")
    else:
        pred = decode(row, "top_k", random.Random(SEED + index))
    return int(pred == gold)


## The concept, built once: logits become text

Temperature uses $p_T(i)=\exp(z_i/T)/\sum_j\exp(z_j/T)$. For lesson logits $[3,2,0.5]$, greedy chooses A.

In [ ]:

logits = [3.0, 2.0, 0.5]
probs = temperature_probs(logits)
winner = TOKENS[int(np.argmax(probs))]
print(np.round(probs, 6), winner)
assert np.allclose(np.round(probs, 6), [0.689672, 0.253716, 0.056612])
assert winner == "A"


Truncation must renormalize: $p_k(i)=p(i)\mathbf{1}\{i\in TopK\}/\sum_{j\in TopK}p(j)$. Beam search compares log scores, not products, so a lesson score such as $-1.203973$ remains numerically safe.

In [ ]:

base = np.array([0.55, 0.25, 0.10, 0.06, 0.04])
filtered = top_k_probs(base, 3)
nucleus, keep = nucleus_probs([0.50, 0.25, 0.12053, 0.08, 0.04947], 0.87)
score = beam_log_score([0.3, 0.3, 0.3])
print("top-k", filtered)
print("nucleus count", len(keep))
print("beam", round(score, 6))
assert np.allclose(filtered[3:], [0.0, 0.0])
assert round(float(filtered[:3].sum()), 6) == 1.0
assert len(keep) == 3
assert round(score, 6) == -3.611918
assert round(math.log(0.3), 6) == -1.203973


## The dataset ladder

D1 is the lesson next-token distribution, then clean rows, near-tie traps, response candidates, and D5 longer generated sequences.

In [ ]:

ladder = make_sequence_ladder("8.15")
for rung in ladder:
    print(rung["name"], "steps=", len(rung["rows"]), "sample logits=", rung["rows"][0])


In [ ]:

rows = []
metrics = []
diversity = []
for rung in ladder:
    successes = [constrained_success(row, gold, i) for i, (row, gold) in enumerate(zip(rung["rows"], rung["target"]))]
    generated = [decode(row, "top_k", random.Random(SEED + i)) for i, row in enumerate(rung["rows"])]
    score = sum(successes) / len(successes)
    metrics.append(score)
    diversity.append(len(set(generated)) / len(generated))
    rows.append([rung["name"], len(rung["rows"]), round(score, 3), round(diversity[-1], 3)])
show_table(rows, ["rung", "steps", "accuracy", "diversity"])


In [ ]:

fig, axes = plt.subplots(1, 6, figsize=(15, 2.4))
for ax, rung in zip(axes[:5], ladder):
    mat = np.vstack([temperature_probs(row) for row in rung["rows"]])
    ax.imshow(mat, cmap="magma", aspect="auto", vmin=0, vmax=1)
    ax.set_title(rung["name"].split()[0])
    ax.set_xlabel("token")
    ax.set_ylabel("step")
axes[5].plot(diversity, metrics, marker="o")
axes[5].set_ylim(0, 1.05)
axes[5].set_title("accuracy vs diversity")
axes[5].set_xlabel("diversity")
fig.tight_layout()


## Pitfall on D5: filtering without normalization and product underflow

Top-k must rescale surviving mass. Beam search should accumulate log probabilities so long sequences do not collapse toward zero.

In [ ]:

d5 = ladder[-1]
probs = temperature_probs(d5["rows"][0])
wrong = np.where(probs >= sorted(probs)[-2], probs, 0.0)
right = top_k_probs(probs, 2)
product_score = float(np.prod([0.01] * 400))
log_score = beam_log_score([0.01] * 400)
print("wrong sum", wrong.sum())
print("right sum", right.sum())
print("product", product_score)
print("log", round(log_score, 3))
assert wrong.sum() < 1.0
assert round(float(right.sum()), 6) == 1.0
assert product_score == 0.0
assert log_score < 0


## Evaluate it + Practice

- Metric: task constraint accuracy; no-skill baseline is greedy on every row.
- Sanity check: probabilities sum to one after every filtering step.
- Ablation: remove renormalization and sampled probabilities are invalid.
- Failure signal: long beam products underflow to zero.

Practice: implement temperature before and after top-k and compare.

Practice: add a repetition penalty to D5 and re-score constraint accuracy.